# 02 — Species Classification (MobileNetV3-Small vs EfficientNet-B0)
**Phase 2.** Identical transfer-learning recipe for both backbones, then comparison, ablation (augmentation), temperature-scaling calibration + ECE, an honest domain-gap slot, and ONNX export of the winner. All logic in `ml/src/pipeline.py`; the full study is `ml/src/run_study.py`. `SEED=42`.

> **v2 update:** this is the Phase-2 **baseline** (EfficientNet-B0, 94.4%). The deployed model was later upgraded to **ConvNeXt-Tiny (98.4%)** via **`03_colab_train_species.ipynb`** (stronger datasets, modern augmentation, Optuna tuning, richer evaluation charts).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)
print('repo root:', ROOT)

In [ ]:
from ml.src import pipeline as P
from ml.src.config import CLASS_NAMES
for a in ['mobilenet_v3_small','efficientnet_b0']:
    mdl = P.build_model(a)
    print(f'{a:22s} params={P.param_count(mdl):,}  size={P.model_size_mb(mdl)}MB  cpu_latency={P.cpu_latency_ms(mdl)}ms')

## Recipe (identical for both)
Two stages: (1) freeze backbone, train the new classifier head; (2) unfreeze and fine-tune at a lower LR. AdamW, cross-entropy, best-val checkpoint. See `P.train_model`. To train from scratch here:
```python
m = P.build_split_manifest(); loaders = P.make_loaders(m, P.TrainConfig())
model, hist = P.train_model('efficientnet_b0', loaders, P.TrainConfig())
```
The full study (both models + ablation + calibration + export) is run once via `python -m ml.src.run_study`; we load its artefacts below.

## Results (loaded from the executed study)

In [ ]:
import json, pandas as pd
from pathlib import Path
mp = Path('ml/exported/metrics.json')
assert mp.exists(), 'Run `python -m ml.src.run_study` first.'
R = json.loads(mp.read_text())
rows = []
for a, mm in R['models'].items():
    rows.append({'model': a + (' (winner)' if a==R['winner'] else ''), 'accuracy': mm['accuracy'],
                 'macro_f1': mm['macro_f1'], 'params': mm['params'], 'size_MB': mm['size_mb'],
                 'cpu_ms': mm['cpu_latency_ms']})
pd.DataFrame(rows)

In [ ]:
print('Ablation (augmentation):', R['ablation'])
print('Calibration:', R['calibration'])
print('Macro ROC-AUC (OvR):', R.get('winner_macro_roc_auc_ovr'))
print('Kathmandu domain gap:', R['kathmandu_domain_gap'])

### Per-class metrics (winner)

In [ ]:
import pandas as pd
pc = R.get('winner_per_class', {})
roc = R.get('winner_roc_auc_per_class', {})
pd.DataFrame([{'class':c, **{k:v[k] for k in ['precision','recall','f1-score','support']}, 'roc_auc':roc.get(c)} for c,v in pc.items()]) if pc else 'run extra_eval'

### Figures

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
from pathlib import Path
figs = ['model_comparison.png', 'class_distribution.png', f"training_curves_{R['winner']}.png",
        f"confusion_{R['winner']}.png", 'confusion_normalized.png', 'per_class_f1.png',
        'roc_curves.png', 'pr_curves.png', 'reliability_diagram.png']
for f in figs:
    p = Path('docs/figures')/f
    if p.exists():
        plt.figure(figsize=(6,4)); plt.imshow(mpimg.imread(p)); plt.axis('off'); plt.title(f); plt.show()

## ONNX export — verify it loads & matches the backend contract

In [ ]:
import numpy as np, onnxruntime as ort
sess = ort.InferenceSession('ml/exported/species_model.onnx', providers=['CPUExecutionProvider'])
x = np.random.randn(1,3,224,224).astype('float32')
out = sess.run(None, {sess.get_inputs()[0].name: x})[0]
print('ONNX output shape:', out.shape, '→ classes:', CLASS_NAMES)
print('Set SPECIES_TEMPERATURE=%s in backend .env (calibrated).' % R['calibration']['temperature'])

**Done.** The winner is exported to `ml/exported/species_model.onnx`; the FastAPI backend auto-loads it (`backend/app/ml/species.py`) and routes low-confidence predictions to `Unverified` → Gemini second opinion.